## Install ultralytics + best Yolov11 model

# Create Classification Crops (YOLOv11 batch cropper)

## Overview
This is a **data-prep utility**, not a training notebook. It runs the trained YOLOv11 detector
(`best.pt`) over a raw, brand-labeled image dataset (`train/valid` x `bharat/hp/indane/unknown`)
and writes out cylinder crops into a matching folder structure. Those crops are the input dataset
for the downstream classifier training notebooks (e.g. `lpg_classifier_v2.ipynb`). Images where
the detector finds zero boxes are copied through uncropped as a "fallback" (and logged), so the
notebook also includes cells to visualize which images the detector missed.

## How to Run
1. **Runtime:** Colab GPU recommended (YOLO inference is much faster), but this is a CPU-tolerable
   utility if GPU isn't available — it's inference-only, not training, so it will just run slower.
2. **Upload:** upload `best.pt` (trained YOLOv11 detector weights) when prompted in the first
   cell. Then upload/provide the raw dataset zip — the notebook expects it unzipped to
   `/content/lpg_dataset/{train,valid}/{bharat,hp,indane,unknown}/`.
3. **Execution order:** run top to bottom. The "Uploading images" cell (cell-3, Option A) is an
   alternate/manual path and is not required if you instead unzip a full dataset archive directly
   in the "Unzip folder" cell — note this cell's output shows it was interrupted
   (`KeyboardInterrupt`) in the saved run, i.e. it was skipped in favor of the zip-based flow.
   The cropping cell must finish before the two fallback-visualization cells (they read files it
   wrote).
4. **Expected outputs:** a full crop dataset under `/content/lpg_crops/{train,valid}/{brand}/`,
   plus `fallback_log.txt` and `fallbacks.png` documenting images the detector failed to find a
   cylinder in.

## Model / Dataset Info
| | |
|---|---|
| Detector used | `best.pt` (YOLOv11), inference-only, `conf=0.4` |
| Input dataset | `/content/lpg_dataset/{train,valid}/{bharat,hp,indane,unknown}` |
| Output dataset | `/content/lpg_crops/{train,valid}/{brand}` — one crop file per detected box (`{stem}_crop{idx}.ext`); multiple cylinders in one source image all get saved |
| Crop counts (this run) | train: bharat 1003, hp 434, indane 925, unknown 127; valid: bharat 251, hp 100, indane 226, unknown 23 (see cell-7 output) |
| Fallback behavior | If YOLO detects 0 boxes, the original uncropped image is copied through and its path logged to `fallback_log.txt` |

## Current Status
The cropping run completed and produced counts for all 8 split/brand combinations. A modest
fallback rate is present in every bucket (detector missed the cylinder in a handful of images per
class — see per-bucket "fallback" counts in cell-7 output); the two visualization cells at the end
exist specifically to review those fallback images. The "Uploading images" cell (cell-3) has a
stale `KeyboardInterrupt` in its saved output and was evidently not used in the final run — the
zip-based unzip path (cell-5) was used instead. No other apparent gaps.


In [ ]:
!pip install ultralytics -q

from google.colab import files
print("Upload best.pt")
uploaded = files.upload()  # upload best.pt

## Uploading images

In [ ]:
import os

# Option A — upload images directly
print("Upload your test images")
uploaded_imgs = files.upload()

# Save to staging folder
os.makedirs("/content/input_images", exist_ok=True)
for fname, data in uploaded_imgs.items():
    with open(f"/content/input_images/{fname}", "wb") as f:
        f.write(data)

print(f"Uploaded {len(uploaded_imgs)} images")

## Unzip folder

In [ ]:
import zipfile
import os

# Unzip the raw brand-labeled dataset (expects train/valid x bharat/hp/indane/unknown inside)
with zipfile.ZipFile("/content/lpg_brand_classification.v1i.folder.zip", "r") as z:  # dataset zip
    z.extractall("/content/lpg_dataset")

print("Unzipped!")

# Check structure
for root, dirs, files_ in os.walk("/content/lpg_dataset"):
    print(f"{root}: {len(files_)} files")
    if len(files_) > 0:
        break

## Running crops using YOLOv11

Batch-runs the detector over every split/brand folder and writes cropped cylinders (or,
for images with no detection, an uncropped fallback copy) to `OUTPUT_BASE`.

In [ ]:
from ultralytics import YOLO
from PIL import Image
import os, shutil

model = YOLO("best.pt")

INPUT_BASE  = "/content/lpg_dataset"   # raw brand-labeled images (train/valid x brand)
OUTPUT_BASE = "/content/lpg_crops"     # cropped output — consumed by classifier notebooks

brands  = ["bharat", "hp", "indane", "unknown"]
splits  = ["train", "valid"]
summary = {}

for split in splits:
    for brand in brands:
        input_folder  = f"{INPUT_BASE}/{split}/{brand}"
        output_folder = f"{OUTPUT_BASE}/{split}/{brand}"

        if not os.path.exists(input_folder):
            print(f"Skipping {split}/{brand} — not found")
            continue

        os.makedirs(output_folder, exist_ok=True)

        images = [f for f in os.listdir(input_folder)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]

        saved   = 0
        skipped = 0

        for fname in images:
          fpath = f"{input_folder}/{fname}"
          try:
              results = model(fpath, conf=0.4, verbose=False)  # lower conf for composites
              boxes   = results[0].boxes

              if len(boxes) == 0:
                  # No detection — fall back to copying the original image through uncropped,
                  # and log it so it can be reviewed later (see fallback visualization cells below)
                  shutil.copy(fpath, f"{output_folder}/{fname}")
                  skipped += 1
                  with open("/content/fallback_log.txt", "a") as log:
                      log.write(f"{split}/{brand}/{fname}\n")
                  continue

              # Save ALL detected cylinders, not just first
              img = Image.open(fpath).convert("RGB")
              for box_idx, box in enumerate(boxes):
                  coords = box.xyxy[0].cpu().numpy()
                  crop = img.crop((
                      max(0, int(coords[0])),
                      max(0, int(coords[1])),
                      min(img.width,  int(coords[2])),
                      min(img.height, int(coords[3]))
                  ))
                  # Add box index to filename to avoid overwriting
                  stem = os.path.splitext(fname)[0]
                  ext  = os.path.splitext(fname)[1]
                  out_fname = f"{stem}_crop{box_idx}{ext}"
                  crop.save(f"{output_folder}/{out_fname}")
                  saved += 1

          except Exception as e:
              print(f"Error {fname}: {e}")
              continue

        summary[f"{split}/{brand}"] = {"cropped": saved, "fallback": skipped}
        print(f"{split}/{brand}: {saved} cropped, {skipped} fallback")

print("\n✅ Done!")

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| Cropped images | `/content/lpg_crops/{train,valid}/{bharat,hp,indane,unknown}/{stem}_crop{idx}.ext` | Per-cylinder crops — the dataset consumed by classifier training notebooks |
| Fallback copies | `/content/lpg_crops/{train,valid}/{brand}/{original filename}` | Uncropped original, copied through when YOLO detected 0 boxes |
| `fallback_log.txt` | `/content/` | One `split/brand/filename` line per fallback image, appended during the crop run |
| `fallbacks.png` | `/content/` (also `files.download()`'d) | Grid visualization of fallback images, for manual review |

The two cells below (fallback visualization) are two independent approaches to the same review
task — one infers fallbacks by comparing file sizes, the other reads `fallback_log.txt` directly.
Both are retained; neither is a true duplicate of the other.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

fallback_images = []

for split in splits:
    for brand in brands:
        input_folder  = f"{INPUT_BASE}/{split}/{brand}"
        output_folder = f"{OUTPUT_BASE}/{split}/{brand}"

        if not os.path.exists(input_folder):
            continue

        input_files  = set(os.listdir(input_folder))
        output_files = set(os.listdir(output_folder))

        # Fallbacks are files where output == input (no crop happened)
        # We can track them by checking file sizes match original
        for fname in input_files:
            in_path  = f"{input_folder}/{fname}"
            out_path = f"{output_folder}/{fname}"
            if os.path.exists(out_path):
                in_size  = os.path.getsize(in_path)
                out_size = os.path.getsize(out_path)
                if in_size == out_size:  # same file = fallback copy
                    fallback_images.append((f"{split}/{brand}", in_path))

print(f"Found {len(fallback_images)} fallbacks\n")

# Display in grid
cols = 4
rows = (len(fallback_images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

for i, (label, path) in enumerate(fallback_images):
    try:
        img = mpimg.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(f"{label}\n{os.path.basename(path)[:20]}", fontsize=8)
        axes[i].axis("off")
    except:
        axes[i].axis("off")

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Fallback Images ({len(fallback_images)} total)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fallbacks.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved fallbacks.png")

In [ ]:
# Read fallback log and display
with open("/content/fallback_log.txt") as f:
    fallback_list = f.read().splitlines()

print(f"Total fallbacks: {len(fallback_list)}")

cols = 4
rows = (len(fallback_list) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i, entry in enumerate(fallback_list):
    parts = entry.split("/")
    split, brand, fname = parts[0], parts[1], parts[2]
    path = f"{INPUT_BASE}/{split}/{brand}/{fname}"
    try:
        img = mpimg.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(f"{split}/{brand}\n{fname[:25]}", fontsize=7)
        axes[i].axis("off")
    except:
        axes[i].axis("off")

for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Fallback Images — {len(fallback_list)} total", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/fallbacks.png", dpi=100, bbox_inches="tight")
plt.show()
files.download("/content/fallbacks.png")